# AkasicDB Hybrid Search: Graph + Full-Text + Vector

In this notebook, you load synthetic product data into AkasicDB and run a single SQL query that combines **graph traversal (Cypher)**, **full-text search (BM25, texton)**, **vector similarity (vectoron)**, and **SQL filters**. Finally, you check the execution plan with PostgreSQL `EXPLAIN`.

## Prerequisites

* An AkasicDB instance
    * Create a Playground instance at [db.akasic.cloud](https://db.akasic.cloud), or
    * Install AkasicDB locally by following the [installation guide](https://docs.graphai.io/akasicdb-docs/latest/en/installation/installation/).
* A connection string
    * Google Colab: click the key icon in the left sidebar and save it as a secret named `DB_DSN`.
    * Other environments: set the `DATABASE_URL` environment variable.
* (Optional) A GPU runtime. In Colab, choose **Runtime → Change runtime type → T4 GPU**. Without a GPU, the notebook uses a smaller dataset and runs on the CPU.

## Step 1: Install dependencies

In [ ]:
%pip install --quiet akasicdb "psycopg[binary]" "sentence-transformers>=2.7" "transformers>=4.51" accelerate

## Step 2: Connect to the database and enable the extension

The notebook uses the `DATABASE_URL` environment variable if it is set. Otherwise, it reads `DB_DSN` from Colab Secrets.

Enable the `akasicdb` extension once per database. `CASCADE` also installs the `vectoron` extension that AkasicDB depends on. Running it again on a database where the extension is already enabled is safe.

In [ ]:
import os

import psycopg
from akasicdb.psycopg import register_vector


def get_dsn() -> str:
    dsn = os.getenv("DATABASE_URL")
    if dsn:
        return dsn
    try:
        from google.colab import userdata
    except ImportError:
        raise RuntimeError("Set the DATABASE_URL environment variable or save DB_DSN in Colab Secrets.")
    return userdata.get("DB_DSN")


conn = psycopg.connect(get_dsn(), autocommit=True)
conn.execute("CREATE EXTENSION IF NOT EXISTS akasicdb CASCADE")
conn.execute("SELECT akasicdb_admin.initialize()")
register_vector(conn)

print(conn.execute("SELECT version()").fetchone()[0])

## Step 3: Create the schema

To make the notebook re-runnable, drop the graph and tables from any previous run first. Product description embeddings are stored as `vectoron.vector(1024)` to match the output dimension of Qwen3-Embedding-0.6B.

In [ ]:
for stmt in (
    "SELECT akasicdb.delete_graph('retail_graph', true)",
    "SELECT akasicdb.undefine_graph('retail_graph')",
):
    try:
        conn.execute(stmt)
    except psycopg.Error:
        pass  # Skip if the graph does not exist.

conn.execute("DROP TABLE IF EXISTS products_std, products, brands CASCADE")

conn.execute("""
CREATE TABLE brands (
  b_id  integer PRIMARY KEY,
  name  text    NOT NULL,
  hot   boolean NOT NULL DEFAULT false
)""")

conn.execute("""
CREATE TABLE products (
  id       integer PRIMARY KEY,
  b_id     integer NOT NULL REFERENCES brands(b_id),
  title    text    NOT NULL,
  category text    NOT NULL,
  price    integer NOT NULL,
  descr    text    NOT NULL,
  emb      vectoron.vector(1024)
)""")
print("ok")

## Step 4: Generate synthetic data

With a GPU, the notebook creates 10,000 products. Without one, it creates 2,000. Ten of the 50 brands are popular (`hot`), and their products are spread evenly across all categories.

Each product description combines a season, an item, and a feature.

In [ ]:
import random

import torch

if torch.cuda.is_available():
    DEVICE, DTYPE, N_PRODUCTS = "cuda", torch.float16, 10_000
else:
    DEVICE, DTYPE, N_PRODUCTS = "cpu", torch.float32, 2_000
print(f"Device: {DEVICE}, products: {N_PRODUCTS:,}")

random.seed(42)
N_BRANDS = 50

seasons = ["summer", "winter", "spring", "fall"]
items = ["running shoes", "sneakers", "sandals", "trekking boots", "slip-ons"]
feats = ["breathable", "lightweight", "well-cushioned", "waterproof"]
cats = ["shoes", "apparel", "bag", "acc", "outdoor"]

templates = [
    lambda s, i, f: f"Looking for {s} {i}? These {f} picks are a great choice.",
    lambda s, i, f: f"{f.capitalize()} {i} made for the {s} season.",
    lambda s, i, f: f"Stay comfortable through the {s} in these {f} {i}.",
]

brands = [(b, f"brand_{b}", b % 5 == 0) for b in range(1, N_BRANDS + 1)]

products = []
for g in range(1, N_PRODUCTS + 1):
    season = seasons[(g // 7) % len(seasons)]
    item = items[(g // 3) % len(items)]
    feat = feats[g % len(feats)]
    descr = templates[(g // 11) % len(templates)](season, item, feat)
    products.append((
        g,                                   # id
        (g // 5) % N_BRANDS + 1,             # b_id: assigned independently of the category
        f"product_{g}",                      # title
        cats[g % len(cats)],                 # category
        10_000 + random.randrange(190_000),  # price
        descr,
    ))

hot_brands = {b for b, _, hot in brands if hot}
hot_shoes = sum(1 for p in products if p[1] in hot_brands and p[3] == "shoes")
print(f"Shoes from popular brands: {hot_shoes}")
for p in products[:3]:
    print(p)

## Step 5: Generate embeddings

Qwen3-Embedding-0.6B is a multilingual embedding model. Encode product descriptions without an instruction, and encode search queries with `prompt_name="query"`. With `normalize_embeddings=True`, ordering by L2 distance (`<->`) is the same as ordering by cosine similarity.

Encoding 10,000 products takes a few minutes on a T4 GPU. Encoding 2,000 products on a CPU takes several minutes or more.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "Qwen/Qwen3-Embedding-0.6B",
    device=DEVICE,
    model_kwargs={"torch_dtype": DTYPE},
    tokenizer_kwargs={"padding_side": "left"},
)

embs = model.encode(
    [p[5] for p in products],
    batch_size=64 if DEVICE == "cuda" else 16,
    normalize_embeddings=True,
    show_progress_bar=True,
)
print(embs.shape)

## Step 6: Load the data

In [ ]:
with conn.cursor() as cur:
    cur.executemany("INSERT INTO brands VALUES (%s, %s, %s)", brands)

    rows = [(*p, emb) for p, emb in zip(products, embs)]
    BATCH = 1000
    for i in range(0, len(rows), BATCH):
        cur.executemany(
            "INSERT INTO products (id, b_id, title, category, price, descr, emb) "
            "VALUES (%s, %s, %s, %s, %s, %s, %s)",
            rows[i:i + BATCH],
        )
        print(f"{min(i + BATCH, len(rows)):,} / {len(rows):,}")

conn.execute("ANALYZE brands, products")
print("done")

## Step 7: Create indexes

Create three kinds of indexes.

* **vectoron**: approximate nearest neighbor (ANN) search on the embeddings
* **texton**: full-text search (BM25) on the product descriptions. This is a whole-row index on `products.*`, and its `mapping` indexes the `descr` field with the language-neutral `standard` analyzer. The `@@` search operator uses this index.
* **B-tree**: the category filter

In [ ]:
conn.execute("CREATE INDEX products_emb_idx ON products USING vectoron (emb vector_l2_ops)")

conn.execute("""
CREATE INDEX products_texton_idx
  ON products
  USING texton ((products.*))
  WITH (mapping='{"fields":{"descr":{"type":"text"}}}')""")

conn.execute("CREATE INDEX products_cat_idx ON products (category)")
print("ok")

## Step 8: Define the graph

Define graph vertices and edges from the existing `products` and `brands` tables. The data is not loaded again.

* Vertices: `v_product` (products) and `v_brand` (brands)
* Edge: `e_made_by` (product → brand)

In [ ]:
conn.execute("SELECT akasicdb.define_graph('retail_graph')")

conn.execute("""
SELECT akasicdb.define_vertex('retail_graph', 'v_product',
  ARRAY['product_id integer', 'title text'],
  'products',
  'SELECT id AS product_id, title FROM products')""")

conn.execute("""
SELECT akasicdb.define_vertex('retail_graph', 'v_brand',
  ARRAY['b_id integer', 'name text', 'hot boolean'],
  'brands',
  'SELECT b_id, name, hot FROM brands')""")

conn.execute("""
SELECT akasicdb.define_edge('retail_graph', 'e_made_by',
  'v_product', 'v_brand',
  null,
  'SELECT null FROM products p, brands b WHERE p.b_id = b.b_id',
  'products p', 'brands b')""")

conn.execute("SELECT akasicdb.create_graph('retail_graph')")
print("graph ready")

## Step 9: Run a hybrid query

A single SQL query combines four conditions.

1. **Graph (Cypher)**: products made by popular brands (`hot = true`)
2. **SQL filter**: `category = 'shoes'`
3. **Full-text search (BM25)**: descriptions that contain both "summer" and "running". `akasicdb.text_score(p)` shows the BM25 score.
4. **Vector similarity**: ordered by semantic similarity to "breathable running shoes for summer"

In [ ]:
q_emb = model.encode(
    ["breathable running shoes for summer"],
    prompt_name="query",
    normalize_embeddings=True,
)[0]

HYBRID_SQL = """
SELECT p.id, p.title, p.price, akasicdb.text_score(p) AS bm25, left(p.descr, 40) AS descr
FROM   products p
JOIN   akasicdb.cypher('retail_graph', $$
           MATCH (pr:v_product)-[:e_made_by]->(b:v_brand)
           WHERE b.hot = true
           RETURN pr.product_id AS product_id
       $$) AS g (product_id integer)
       ON g.product_id = p.id                          -- graph (Cypher)
WHERE  p.category = 'shoes'                            -- SQL filter
  AND  p @@ akasicdb.text_query('summer AND running')  -- full-text search (BM25)
ORDER  BY p.emb <-> %s::vectoron.vector                 -- vector similarity (L2)
LIMIT  10
"""

rows = conn.execute(HYBRID_SQL, (q_emb,)).fetchall()
for row in rows:
    print(row)
print(f"Hybrid results: {len(rows)}")

## Step 10: Check the execution plan

Use standard PostgreSQL `EXPLAIN` to see the plan for the hybrid query.

In [ ]:
plan = conn.execute("EXPLAIN (ANALYZE, BUFFERS) " + HYBRID_SQL, (q_emb,)).fetchall()
for (line,) in plan:
    print(line if len(line) <= 160 else line[:160] + " ...")

### How to read the plan

1. **Full-text search**: `Index Scan using products_texton_idx on products p` means the BM25 search ran on the texton index. A texton `@@` condition is always evaluated through this index scan, and the `category` condition is applied as a `Filter` on the same node.
2. **Vector ordering**: A `Sort` node on the `p.emb <-> ...` distance means the planner computed exact distances only for the candidates that passed the earlier conditions. The planner chooses this path when full-text and graph conditions narrow the candidates a lot.
3. **Graph**: The Cypher query expands into joins over the graph topology tables (`v_product_topo`, `v_brand_topo`, `e_made_by_topo`), so it is optimized together with the other conditions in a single plan.

Try changing the conditions (`WHERE b.hot`, `category`, the full-text query) and compare how the plan changes.